# Table v2 no 1

### Functions

In [ ]:
import pandas as pd

all_pr_df = pd.read_parquet("Data/all_pull_request.parquet")
human_pr_df_without_bots = pd.read_parquet("Data/human_pull_request_removed_bot_prs.parquet")
human_repos_urls_df_removed = pd.DataFrame({'repo_url' : human_pr_df_without_bots['repo_url'].unique()})
all_prs_from_repos_with_human_prs = all_pr_df.merge(human_repos_urls_df_removed, on='repo_url')

pr_comments_df = pd.read_parquet("Data/pr_comments.parquet")
pr_comments_from_repos_with_human_df = pr_comments_df[
    pr_comments_df['pr_id'].isin(all_prs_from_repos_with_human_prs['id'])
]

In [ ]:
human_pr_comments_df = pd.read_parquet("Data/human_pr_comments.parquet")
human_pr_comments_from_repos_with_human_df = human_pr_comments_df[
    human_pr_comments_df['pr_id'].isin(human_pr_df_without_bots['id'])
]

In [ ]:
def compute_count_1(df):
    count_prs = df["pr_id"].nunique()

    count_reviewers_agentic = (
        df[
            df["user_type"].isin(["Bot", "BotOther"])
        ]["user"].nunique()
    )

    count_reviewers_human = (
        df[
            df["user_type"] == "User"
        ]["user"].nunique()
    )

    count_reviews_agentic = len(df[
        df["user_type"].isin(["Bot", "BotOther"])
    ])

    count_reviews_human = len(df[
        df["user_type"] == "User"
    ])

    print("Count # PRs: \t\t\t", count_prs)
    print("Count # Reviewers - agentic: \t", count_reviewers_agentic)
    print("Count # Reviewers - human: \t", count_reviewers_human)
    print("Count # Reviews - agentic: \t", count_reviews_human)
    print("Count # Reviews - human: \t", count_reviews_agentic)

    return count_prs, count_reviewers_agentic, count_reviewers_human, count_reviews_human, count_reviews_agentic

In [ ]:
def compute_mean_std_1(df):
    all_reviews_per_pr = df.groupby("pr_id").size()
    all_mean = all_reviews_per_pr.mean()
    all_std  = all_reviews_per_pr.std()

    agentic_reviewers_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")["user"]
        .nunique()
    )
    agentic_rev_mean = agentic_reviewers_per_pr.mean()
    agentic_rev_std  = agentic_reviewers_per_pr.std()

    human_reviewers_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")["user"]
        .nunique()
    )
    human_rev_mean = human_reviewers_per_pr.mean()
    human_rev_std  = human_reviewers_per_pr.std()

    agentic_reviews_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")
        .size()
    )
    agentic_mean = agentic_reviews_per_pr.mean()
    agentic_std  = agentic_reviews_per_pr.std()

    human_reviews_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")
        .size()
    )
    human_mean = human_reviews_per_pr.mean()
    human_std  = human_reviews_per_pr.std()

    print("Mean + std PRs: \t\t", f"{all_mean:.2f} ± {all_std:.2f}")
    print("Mean + std Reviewers - agentic: ", f"{agentic_rev_mean:.2f} ± {agentic_rev_std:.2f}")
    print("Mean + std Reviewers - human: \t", f"{human_rev_mean:.2f} ± {human_rev_std:.2f}")
    print("Mean + std Reviews - agentic: \t", f"{agentic_mean:.2f} ± {agentic_std:.2f}")
    print("Mean + std Reviews - human: \t", f"{human_mean:.2f} ± {human_std:.2f}")

    return all_mean, all_std, agentic_rev_mean, agentic_rev_std, human_rev_mean, human_rev_std, agentic_mean, agentic_std, human_mean, human_std

In [ ]:
import numpy as np

def median_ci(series, confidence=0.95, n_bootstrap=5000, random_state=42):
    random = np.random.default_rng(random_state)
    data = series.dropna().values
    n = len(data)

    boot_medians = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = random.choice(data, size=n, replace=True)
        boot_medians[i] = np.median(sample)

    alpha = (1 - confidence) / 2
    lower = np.quantile(boot_medians, alpha)
    upper = np.quantile(boot_medians, 1 - alpha)

    return np.median(data), lower, upper

def compute_median_ci_1(df):
    all_reviews_per_pr = df.groupby("pr_id").size()
    med_all, ci_all_low, ci_all_high = median_ci(all_reviews_per_pr)

    agentic_reviewers_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")["user"]
        .nunique()
    )
    med_agentic_reviewers, ci_ar_low, ci_ar_high = median_ci(agentic_reviewers_per_pr)

    human_reviewers_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")["user"]
        .nunique()
    )
    med_human_reviewers, ci_hr_low, ci_hr_high = median_ci(human_reviewers_per_pr)

    agentic_reviews_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")
        .size()
    )
    med_agentic_reviews, ci_arev_low, ci_arev_high = median_ci(agentic_reviews_per_pr)


    human_reviews_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")
        .size()
    
    )
    med_human_reviews, ci_hrev_low, ci_hrev_high = median_ci(human_reviews_per_pr)

    print("Median + CI PRs: \t\t", f"{med_all:.2f} + [{ci_all_low:.2f} - {ci_all_high:.2f}]")
    print("Median + CI Reviewers - agentic: ", f"{med_agentic_reviewers:.2f} + [{ci_ar_low:.2f} - {ci_ar_high:.2f}]")
    print("Median + CI Reviewers - human: \t", f"{med_human_reviewers:.2f} + [{ci_hr_low:.2f} - {ci_hr_high:.2f}]")
    print("Median + CI Reviews - agentic: \t", f"{med_agentic_reviews:.2f} + [{ci_arev_low:.2f} - {ci_arev_high:.2f}]")
    print("Median + CI Reviews - human: \t", f"{med_human_reviews:.2f} + [{ci_hrev_low:.2f} - {ci_hrev_high:.2f}]")

    return med_all, ci_all_low, ci_all_high, med_agentic_reviewers, ci_ar_low, ci_ar_high, med_human_reviewers, ci_hr_low, ci_hr_high, \
        med_agentic_reviews, ci_arev_low, ci_arev_high, med_human_reviews, ci_hrev_low, ci_hrev_high

### pr_comments

In [ ]:
import pandas as pd

In [ ]:
index = pd.MultiIndex.from_tuples(
    [
        ("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"),
        ("$C(P_{AI}| R)$", "# PRs", "-"),
        ("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"),
        ("$C(P_{AI}| R)$", "\\# Reviewers", "Human"),
        ("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"),
        ("$C(P_{AI}| R)$", "\\# Reviews", "Human"),
        ("$C(P_{human}| R)$", "# PRs", "-"),
        ("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"),
        ("$C(P_{human}| R)$", "\\# Reviewers", "Human"),
        ("$C(P_{human}| R)$", "\\# Reviews", "Agentic"),
        ("$C(P_{human}| R)$", "\\# Reviews", "Human"),
    ],
    names=["Dataset", "Category", "Type"]
)

columns = [
    "Count",
    "Mean ± Std",
    "Median, CI"
]

table1 = pd.DataFrame(np.nan, index=index, columns=columns)
table1["Count"] = table1["Count"].astype("Int64")

In [ ]:
t1_prc_count_prs, \
t1_prc_count_reviewers_agentic, t1_prc_count_reviewers_human, \
t1_prc_count_reviews_agentic, t1_prc_count_reviews_human = compute_count_1(pr_comments_df)

Count # PRs: 			 12975
Count # Reviewers - agentic: 	 129
Count # Reviewers - human: 	 2224
Count # Reviews - agentic: 	 11706
Count # Reviews - human: 	 27416


In [ ]:
len(pr_comments_df["id"].unique())

In [13]:
t1_prc_ms_prs_mean, t1_prc_ms_prs_std, \
t1_prc_ms_reviewers_agentic_mean, t1_prc_ms_reviewers_agentic_std, \
t1_prc_ms_reviewers_human_mean, t1_prc_ms_reviewers_human_std, \
t1_prc_ms_reviews_agentic_mean, t1_prc_ms_reviews_agentic_std, \
t1_prc_ms_reviews_human_mean, t1_prc_ms_reviews_human_std = compute_mean_std_1(pr_comments_df)

Mean + std PRs: 		 3.02 ± 3.16
Mean + std Reviewers - agentic:  1.68 ± 1.05
Mean + std Reviewers - human: 	 1.25 ± 0.67
Mean + std Reviews - agentic: 	 2.35 ± 1.95
Mean + std Reviews - human: 	 2.19 ± 2.37


In [14]:
t1_prc_me_ci_prs, t1_prc_me_ci_prs_low, t1_prc_me_ci_prs_high, \
t1_prc_me_ci_reviewers_agentic, t1_prc_me_ci_reviewers_agentic_low, t1_prc_me_ci_reviewers_agentic_high, \
t1_prc_me_ci_reviewers_human, t1_prc_me_ci_reviewers_human_low, t1_prc_me_ci_reviewers_human_high, \
t1_prc_me_ci_reviews_agentic, t1_prc_me_ci_reviews_agentic_low, t1_prc_me_ci_reviews_agentic_high, \
t1_prc_me_ci_reviews_human, t1_prc_me_ci_reviews_human_low, t1_prc_me_ci_reviews_human_high = compute_median_ci_1(pr_comments_df)

Median + CI PRs: 		 2.00 + [2.00 - 2.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 2.00 + [2.00 - 2.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


In [15]:
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"), "Count"] = t1_prc_count_prs
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Count"] = t1_prc_count_reviewers_agentic
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Count"] = t1_prc_count_reviewers_human
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Count"] = t1_prc_count_reviews_agentic
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Count"] = t1_prc_count_reviews_human

table1.loc[("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"), "Mean ± Std"] = f"{t1_prc_ms_prs_mean:.2f} ± {t1_prc_ms_prs_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_prc_ms_reviewers_agentic_mean:.2f} ± {t1_prc_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_prc_ms_reviewers_human_mean:.2f} ± {t1_prc_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_prc_ms_reviews_agentic_mean:.2f} ± {t1_prc_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_prc_ms_reviews_human_mean:.2f} ± {t1_prc_ms_reviews_human_std:.2f}"

table1.loc[("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"), "Median, CI"] = f"{t1_prc_me_ci_prs:.2f} + [{t1_prc_me_ci_prs_low:.2f} - {t1_prc_me_ci_prs_high:.2f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_prc_me_ci_reviewers_agentic:.2f} + [{t1_prc_me_ci_reviewers_agentic_low:.2f} - {t1_prc_me_ci_reviewers_agentic_high:.2f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_prc_me_ci_reviewers_human:.2f} + [{t1_prc_me_ci_reviewers_human_low:.2f} - {t1_prc_me_ci_reviewers_human_high:.2f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_prc_me_ci_reviews_agentic:.2f} + [{t1_prc_me_ci_reviews_agentic_low:.2f} - {t1_prc_me_ci_reviews_agentic_high:.2f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_prc_me_ci_reviews_human:.2f} + [{t1_prc_me_ci_reviews_human_low:.2f} - {t1_prc_me_ci_reviews_human_high:.2f}]"

table1


/tmp/ipykernel_31851/1499038631.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.02 ± 3.16' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table1.loc[("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"), "Mean ± Std"] = f"{t1_prc_ms_prs_mean:.2f} ± {t1_prc_ms_prs_std:.2f}"
/tmp/ipykernel_31851/1499038631.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.00 + [2.00 - 2.00]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table1.loc[("$C(P_{AI}\\mid R_{pop})$", "# PRs", "-"), "Median, CI"] = f"{t1_prc_me_ci_prs:.2f} + [{t1_prc_me_ci_prs_low:.2f} - {t1_prc_me_ci_prs_high:.2f}]"


Count   Mean ± Std  \
Dataset                 Category     Type                          
$C(P_{AI}\mid R_{pop})$ # PRs        -        12975  3.02 ± 3.16   
                        \# Reviewers Agentic    129  1.68 ± 1.05   
                                     Human     2224  1.25 ± 0.67   
                        \# Reviews   Agentic  11706  2.35 ± 1.95   
                                     Human    27416  2.19 ± 2.37   
$C(P_{AI}| R)$          # PRs        -         <NA>          NaN   
                        \# Reviewers Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   
                        \# Reviews   Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   
$C(P_{human}| R)$       # PRs        -         <NA>          NaN   
                        \# Reviewers Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   
                        \# Reviews   Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   

                                                        Median, CI  
Dataset                 Category     Type                           
$C(P_{AI}\mid R_{pop})$ # PRs        -        2.00 + [2.00 - 2.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  2.00 + [2.00 - 2.00]  
                                     Human    1.00 + [1.00 - 1.00]  
$C(P_{AI}| R)$          # PRs        -                         NaN  
                        \# Reviewers Agentic                   NaN  
                                     Human                     NaN  
                        \# Reviews   Agentic                   NaN  
                                     Human                     NaN  
$C(P_{human}| R)$       # PRs        -                         NaN  
                        \# Reviewers Agentic                   NaN  
                                     Human                     NaN  
                        \# Reviews   Agentic                   NaN  
                                     Human                     NaN

In [16]:
!pip install jinja2

### pr_comments_from_repos_with_human_df

In [17]:
t1_prc_rwh_count_prs, \
t1_prc_rwh_count_reviewers_agentic, t1_prc_rwh_count_reviewers_human, \
t1_prc_rwh_count_reviews_agentic, t1_prc_rwh_count_reviews_human = compute_count_1(pr_comments_from_repos_with_human_df)

Count # PRs: 			 6835
Count # Reviewers - agentic: 	 73
Count # Reviewers - human: 	 1147
Count # Reviews - agentic: 	 6685
Count # Reviews - human: 	 16396


In [19]:
t1_prc_rwh_ms_prs_mean, t1_prc_rwh_ms_prs_std, \
t1_prc_rwh_ms_reviewers_agentic_mean, t1_prc_rwh_ms_reviewers_agentic_std, \
t1_prc_rwh_ms_reviewers_human_mean, t1_prc_rwh_ms_reviewers_human_std, \
t1_prc_rwh_ms_reviews_agentic_mean, t1_prc_rwh_ms_reviews_agentic_std, \
t1_prc_rwh_ms_reviews_human_mean, t1_prc_rwh_ms_rwh_reviews_human_std = compute_mean_std_1(pr_comments_from_repos_with_human_df)

Mean + std PRs: 		 3.38 ± 3.38
Mean + std Reviewers - agentic:  1.87 ± 1.21
Mean + std Reviewers - human: 	 1.26 ± 0.75
Mean + std Reviews - agentic: 	 2.67 ± 2.16
Mean + std Reviews - human: 	 2.16 ± 2.41


In [20]:
t1_prc_rwh_me_ci_prs, t1_prc_rwh_me_ci_prs_low, t1_prc_rwh_me_ci_prs_high, \
t1_prc_rwh_me_ci_reviewers_agentic, t1_prc_rwh_me_ci_reviewers_agentic_low, t1_prc_rwh_me_ci_reviewers_agentic_high, \
t1_prc_rwh_me_ci_reviewers_human, t1_prc_rwh_me_ci_reviewers_human_low, t1_prc_rwh_me_ci_reviewers_human_high, \
t1_prc_rwh_me_ci_reviews_agentic, t1_prc_rwh_me_ci_reviews_agentic_low, t1_prc_rwh_me_ci_reviews_agentic_high, \
t1_prc_rwh_me_ci_reviews_human, t1_prc_rwh_me_ci_reviews_human_low, t1_prc_rwh_me_ci_reviews_human_high = compute_median_ci_1(pr_comments_from_repos_with_human_df)

Median + CI PRs: 		 2.00 + [2.00 - 3.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 2.00 + [2.00 - 2.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


In [21]:
table1.loc[("$C(P_{AI}| R)$", "# PRs", "-"), "Count"] = t1_prc_rwh_count_prs
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Count"] = t1_prc_rwh_count_reviewers_agentic
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Count"] = t1_prc_rwh_count_reviewers_human
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Count"] = t1_prc_rwh_count_reviews_agentic
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Count"] = t1_prc_rwh_count_reviews_human

table1.loc[("$C(P_{AI}| R)$", "# PRs", "-"), "Mean ± Std"] = f"{t1_prc_rwh_ms_prs_mean:.2f} ± {t1_prc_rwh_ms_prs_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviewers_agentic_mean:.2f} ± {t1_prc_rwh_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviewers_human_mean:.2f} ± {t1_prc_rwh_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviews_agentic_mean:.2f} ± {t1_prc_rwh_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviews_human_mean:.2f} ± {t1_prc_rwh_ms_rwh_reviews_human_std:.2f}"

table1.loc[("$C(P_{AI}| R)$", "# PRs", "-"), "Median, CI"] = f"{t1_prc_rwh_me_ci_prs:.2f} + [{t1_prc_rwh_me_ci_prs_low:.2f} - {t1_prc_rwh_me_ci_prs_high:.2f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviewers_agentic:.2f} + [{t1_prc_rwh_me_ci_reviewers_agentic_low:.2f} - {t1_prc_rwh_me_ci_reviewers_agentic_high:.2f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviewers_human:.2f} + [{t1_prc_rwh_me_ci_reviewers_human_low:.2f} - {t1_prc_rwh_me_ci_reviewers_human_high:.2f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviews_agentic:.2f} + [{t1_prc_rwh_me_ci_reviews_agentic_low:.2f} - {t1_prc_rwh_me_ci_reviews_agentic_high:.2f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviews_human:.2f} + [{t1_prc_rwh_me_ci_reviews_human_low:.2f} - {t1_prc_rwh_me_ci_reviews_human_high:.2f}]"

table1

Count   Mean ± Std  \
Dataset                 Category     Type                          
$C(P_{AI}\mid R_{pop})$ # PRs        -        12975  3.02 ± 3.16   
                        \# Reviewers Agentic    129  1.68 ± 1.05   
                                     Human     2224  1.25 ± 0.67   
                        \# Reviews   Agentic  11706  2.35 ± 1.95   
                                     Human    27416  2.19 ± 2.37   
$C(P_{AI}| R)$          # PRs        -         6835  3.38 ± 3.38   
                        \# Reviewers Agentic     73  1.87 ± 1.21   
                                     Human     1147  1.26 ± 0.75   
                        \# Reviews   Agentic   6685  2.67 ± 2.16   
                                     Human    16396  2.16 ± 2.41   
$C(P_{human}| R)$       # PRs        -         <NA>          NaN   
                        \# Reviewers Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   
                        \# Reviews   Agentic   <NA>          NaN   
                                     Human     <NA>          NaN   

                                                        Median, CI  
Dataset                 Category     Type                           
$C(P_{AI}\mid R_{pop})$ # PRs        -        2.00 + [2.00 - 2.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  2.00 + [2.00 - 2.00]  
                                     Human    1.00 + [1.00 - 1.00]  
$C(P_{AI}| R)$          # PRs        -        2.00 + [2.00 - 3.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  2.00 + [2.00 - 2.00]  
                                     Human    1.00 + [1.00 - 1.00]  
$C(P_{human}| R)$       # PRs        -                         NaN  
                        \# Reviewers Agentic                   NaN  
                                     Human                     NaN  
                        \# Reviews   Agentic                   NaN  
                                     Human                     NaN

### human_pr_comments_from_repos_with_human_df

In [22]:
t1_hprc_rwh_count_prs, \
t1_hprc_rwh_count_reviewers_agentic, t1_hprc_rwh_count_reviewers_human, \
t1_hprc_rwh_count_reviews_agentic, t1_hprc_rwh_count_reviews_human = compute_count_1(human_pr_comments_from_repos_with_human_df)

Count # PRs: 			 3650
Count # Reviewers - agentic: 	 82
Count # Reviewers - human: 	 1413
Count # Reviews - agentic: 	 4937
Count # Reviews - human: 	 5064


In [23]:
t1_hprc_rwh_ms_prs_mean, t1_hprc_rwh_ms_prs_std, \
t1_hprc_rwh_ms_reviewers_agentic_mean, t1_hprc_rwh_ms_reviewers_agentic_std, \
t1_hprc_rwh_ms_reviewers_human_mean, t1_hprc_rwh_ms_reviewers_human_std, \
t1_hprc_rwh_ms_reviews_agentic_mean, t1_hprc_rwh_ms_reviews_agentic_std, \
t1_hprc_rwh_ms_reviews_human_mean, t1_hprc_rwh_ms_rwh_reviews_human_std = compute_mean_std_1(human_pr_comments_from_repos_with_human_df)

Mean + std PRs: 		 2.74 ± 2.99
Mean + std Reviewers - agentic:  1.63 ± 0.99
Mean + std Reviewers - human: 	 1.51 ± 0.85
Mean + std Reviews - agentic: 	 2.00 ± 1.66
Mean + std Reviews - human: 	 2.40 ± 3.16


In [24]:
t1_hprc_rwh_me_ci_prs, t1_hprc_rwh_me_ci_prs_low, t1_hprc_rwh_me_ci_prs_high, \
t1_hprc_rwh_me_ci_reviewers_agentic, t1_hprc_rwh_me_ci_reviewers_agentic_low, t1_hprc_rwh_me_ci_reviewers_agentic_high, \
t1_hprc_rwh_me_ci_reviewers_human, t1_hprc_rwh_me_ci_reviewers_human_low, t1_hprc_rwh_me_ci_reviewers_human_high, \
t1_hprc_rwh_me_ci_reviews_agentic, t1_hprc_rwh_me_ci_reviews_agentic_low, t1_hprc_rwh_me_ci_reviews_agentic_high, \
t1_hprc_rwh_me_ci_reviews_human, t1_hprc_rwh_me_ci_reviews_human_low, t1_hprc_rwh_me_ci_reviews_human_high = compute_median_ci_1(human_pr_comments_from_repos_with_human_df)

Median + CI PRs: 		 2.00 + [2.00 - 2.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


In [25]:
table1.loc[("$C(P_{human}| R)$", "# PRs", "-"), "Count"] = t1_hprc_rwh_count_prs
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Count"] = t1_hprc_rwh_count_reviewers_agentic
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Count"] = t1_hprc_rwh_count_reviewers_human
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Count"] = t1_hprc_rwh_count_reviews_agentic
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Count"] = t1_hprc_rwh_count_reviews_human

table1.loc[("$C(P_{human}| R)$", "# PRs", "-"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_prs_mean:.2f} ± {t1_hprc_rwh_ms_prs_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviewers_agentic_mean:.2f} ± {t1_hprc_rwh_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviewers_human_mean:.2f} ± {t1_hprc_rwh_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviews_agentic_mean:.2f} ± {t1_hprc_rwh_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviews_human_mean:.2f} ± {t1_hprc_rwh_ms_rwh_reviews_human_std:.2f}"

table1.loc[("$C(P_{human}| R)$", "# PRs", "-"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_prs:.2f} + [{t1_hprc_rwh_me_ci_prs_low:.2f} - {t1_hprc_rwh_me_ci_prs_high:.2f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviewers_agentic:.2f} + [{t1_hprc_rwh_me_ci_reviewers_agentic_low:.2f} - {t1_hprc_rwh_me_ci_reviewers_agentic_high:.2f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviewers_human:.2f} + [{t1_hprc_rwh_me_ci_reviewers_human_low:.2f} - {t1_hprc_rwh_me_ci_reviewers_human_high:.2f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviews_agentic:.2f} + [{t1_hprc_rwh_me_ci_reviews_agentic_low:.2f} - {t1_hprc_rwh_me_ci_reviews_agentic_high:.2f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviews_human:.2f} + [{t1_hprc_rwh_me_ci_reviews_human_low:.2f} - {t1_hprc_rwh_me_ci_reviews_human_high:.2f}]"

table1

Count   Mean ± Std  \
Dataset                 Category     Type                          
$C(P_{AI}\mid R_{pop})$ # PRs        -        12975  3.02 ± 3.16   
                        \# Reviewers Agentic    129  1.68 ± 1.05   
                                     Human     2224  1.25 ± 0.67   
                        \# Reviews   Agentic  11706  2.35 ± 1.95   
                                     Human    27416  2.19 ± 2.37   
$C(P_{AI}| R)$          # PRs        -         6835  3.38 ± 3.38   
                        \# Reviewers Agentic     73  1.87 ± 1.21   
                                     Human     1147  1.26 ± 0.75   
                        \# Reviews   Agentic   6685  2.67 ± 2.16   
                                     Human    16396  2.16 ± 2.41   
$C(P_{human}| R)$       # PRs        -         3650  2.74 ± 2.99   
                        \# Reviewers Agentic     82  1.63 ± 0.99   
                                     Human     1413  1.51 ± 0.85   
                        \# Reviews   Agentic   4937  2.00 ± 1.66   
                                     Human     5064  2.40 ± 3.16   

                                                        Median, CI  
Dataset                 Category     Type                           
$C(P_{AI}\mid R_{pop})$ # PRs        -        2.00 + [2.00 - 2.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  2.00 + [2.00 - 2.00]  
                                     Human    1.00 + [1.00 - 1.00]  
$C(P_{AI}| R)$          # PRs        -        2.00 + [2.00 - 3.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  2.00 + [2.00 - 2.00]  
                                     Human    1.00 + [1.00 - 1.00]  
$C(P_{human}| R)$       # PRs        -        2.00 + [2.00 - 2.00]  
                        \# Reviewers Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]  
                        \# Reviews   Agentic  1.00 + [1.00 - 1.00]  
                                     Human    1.00 + [1.00 - 1.00]

In [26]:
#table1_reset = table1.reset_index()
latex = table1.to_latex(
    multirow=True,
    escape=False,
    index_names=True,
    header=True,       
)
latex = latex.replace("# PRs & -", r"\multicolumn{2}{l}{\# PRs}")
print(latex)


\begin{tabular}{lllrll}
\toprule
 &  &  & Count & Mean ± Std & Median, CI \\
Dataset & Category & Type &  &  &  \\
\midrule
\multirow[t]{5}{*}{$C(P_{AI}\mid R_{pop})$} & \multicolumn{2}{l}{\# PRs} & 12975 & 3.02 ± 3.16 & 2.00 + [2.00 - 2.00] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviewers} & Agentic & 129 & 1.68 ± 1.05 & 1.00 + [1.00 - 1.00] \\
 &  & Human & 2224 & 1.25 ± 0.67 & 1.00 + [1.00 - 1.00] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviews} & Agentic & 11706 & 2.35 ± 1.95 & 2.00 + [2.00 - 2.00] \\
 &  & Human & 27416 & 2.19 ± 2.37 & 1.00 + [1.00 - 1.00] \\
\cline{1-6} \cline{2-6}
\multirow[t]{5}{*}{$C(P_{AI}| R)$} & \multicolumn{2}{l}{\# PRs} & 6835 & 3.38 ± 3.38 & 2.00 + [2.00 - 3.00] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviewers} & Agentic & 73 & 1.87 ± 1.21 & 1.00 + [1.00 - 1.00] \\
 &  & Human & 1147 & 1.26 ± 0.75 & 1.00 + [1.00 - 1.00] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviews} & Agentic & 6685 & 2.67 ± 2.16 & 2.00 + [2.00 - 2.00] \\
 &  & Human & 16396 & 2.16 ±